# 20 Activation Checkpoint 重计算如何在显存与算力间权衡？

## 面试回答主线

activation checkpoint 的本质是前向少保存中间激活，反向需要时从 checkpoint 输入重算对应分段，以额外 FLOPs 换显存。它不会减少参数或 optimizer state，且必须保证重算时的随机性、dropout、mixed precision 和通信语义一致。面试中应量化“保存了多少激活”和“多做了多少前向层”，而不是只说省显存。实验用八层手写特征变换比较保存全部激活与每四层一个 checkpoint 的保存数量、重算层数，并用随机 dropout mask 不可复现造成梯度不一致作为失败。

**核心公式：** 若 $L$ 层每层激活为 $A$，全保存约为 $O(LA)$；分为 $K$ 段 checkpoint 时保存约为 $O(KA)$，代价是反向阶段重算各段前向。最优分段还取决于层异质性。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
class FeatureStack(nn.Module):  # 定义可逐层调用的八层特征变换栈。
    def __init__(self, depth=8):  # 创建固定深度的参数矩阵。
        super().__init__()  # 初始化模块父类。
        self.weights = nn.ParameterList([nn.Parameter(torch.randn(3, 3) * 0.18) for _ in range(depth)])  # 创建每层线性变换参数。
    def layer(self, value, index):  # 明确实现单层前向变换。
        return torch.tanh(value @ self.weights[index])  # 返回一层非线性特征。
    def forward(self, value):  # 提供完整前向以满足模块接口。
        for index in range(len(self.weights)):  # 依次遍历全部层。
            value = self.layer(value, index)  # 更新隐藏表示。
        return value  # 返回最终表示。
stack = FeatureStack()  # 创建八层教学网络。
saved_activations = [features]  # 全保存策略先保存输入激活。
value = features  # 初始化逐层前向状态。
for index in range(8):  # 执行八层前向。
    value = stack.layer(value, index)  # 计算当前层激活。
    saved_activations.append(value.detach().clone())  # 保存每层激活以模拟常规反向。
baseline_metric = len(saved_activations)  # 记录全保存的激活份数。
print(f'全保存：激活份数={baseline_metric}，最终特征 RMS={float(value.pow(2).mean().sqrt()):.4f}')  # 展示基线内存代理指标。


全保存：激活份数=9，最终特征 RMS=0.0000


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
checkpoint_inputs = [features.detach().clone()]  # 只保存第一个 checkpoint 输入。
value = features  # 重新开始前向状态。
for index in range(8):  # 执行相同八层前向。
    value = stack.layer(value, index)  # 计算当前层特征。
    if (index + 1) % 4 == 0:  # 每四层保存一次 checkpoint。
        checkpoint_inputs.append(value.detach().clone())  # 保存分段边界激活而非每层激活。
recomputed_layers = 0  # 统计反向时需要额外执行的层数。
for start in [0, 4]:  # 模拟从两个 checkpoint 输入重算两个分段。
    replay = checkpoint_inputs[start // 4]  # 读取对应分段的 checkpoint 输入。
    for index in range(start, start + 4):  # 重放该分段的四层前向。
        replay = stack.layer(replay, index)  # 实际执行重计算而非只计数。
        recomputed_layers += 1  # 累加重算层数。
core_metric = len(checkpoint_inputs)  # 记录 checkpoint 策略保存的激活份数。
print(f'每四层 checkpoint：保存激活={core_metric}，反向重算层={recomputed_layers}，最终重放 RMS={float(replay.pow(2).mean().sqrt()):.4f}')  # 展示显存与算力交换。


每四层 checkpoint：保存激活=3，反向重算层=8，最终重放 RMS=0.0000


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=9.000000
核心机制     | 指标=3.000000


## 结果解读

基线和核心输出只在本受控案例中比较。生产实现必须与 FSDP、pipeline、flash attention 和 RNG state 协同；需实测 peak memory、tokens/s 和重算比例，而不是只套理论。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
torch.manual_seed(31)  # 固定第一次 dropout 的随机状态。
first_mask = (torch.rand_like(features) > 0.5).float()  # 生成前向阶段的 dropout mask。
forward_value = features * first_mask  # 应用前向阶段随机 mask。
torch.manual_seed(32)  # 故意在重算前改变随机状态。
replay_mask = (torch.rand_like(features) > 0.5).float()  # 生成不一致的重算 mask。
failure_metric = float((features * replay_mask - forward_value).abs().sum())  # 测量前向与重算的差异。
torch.manual_seed(31)  # 恢复与前向相同的 RNG 状态。
fixed_mask = (torch.rand_like(features) > 0.5).float()  # 重新生成一致的 mask。
fix_metric = float((features * fixed_mask - forward_value).abs().sum())  # 测量恢复 RNG 后的差异。
print(f'失败：RNG 未恢复时重算差异={failure_metric:.3f}；修复：恢复 RNG 后差异={fix_metric:.3f}')  # 展示随机性契约。


失败：RNG 未恢复时重算差异=4.000；修复：恢复 RNG 后差异=0.000


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产实现必须与 FSDP、pipeline、flash attention 和 RNG state 协同；需实测 peak memory、tokens/s 和重算比例，而不是只套理论。

**常见坑：** 重算时 dropout RNG 没有恢复，或重算函数有副作用/缓存写入，导致前反向不一致。

**延伸追问：** 为什么 checkpoint 边界通常放在 Transformer block 而非任意算子？如何和 selective activation checkpoint、sequence parallel 一起评估？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert baseline_metric == 9  # 验证八层全保存包含输入和八份激活。
assert core_metric == 3  # 验证每四层 checkpoint 仅保存输入、四层和八层边界。
assert recomputed_layers == 8  # 验证反向阶段实际重算了两个完整分段。
assert failure_metric > fix_metric  # 验证恢复 RNG 消除了重算随机差异。
